In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/Dockerfile
FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y \
    git \
    curl \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .

RUN pip install --upgrade pip

RUN pip install --no-cache-dir \
    torch==2.3.1 \
    torchvision==0.18.1 \
    --index-url https://download.pytorch.org/whl/cpu

RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY .env.example .env

EXPOSE 8080

CMD exec uvicorn src.api.main:app --host 0.0.0.0 --port ${PORT:-8080}

Overwriting /content/drive/MyDrive/isic-flagship-project/Dockerfile


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/requirements.txt
fastapi==0.115.2
uvicorn[standard]==0.32.0
pydantic==2.9.2
pydantic-settings==2.6.1
sqlalchemy==2.0.35
alembic==1.13.2
aiosqlite==0.20.0
python-dotenv==1.0.1
pandas==2.2.2
numpy==1.26.4
pillow==10.4.0
timm==1.0.11
lightgbm==4.5.0
catboost==1.2.7
xgboost==2.1.1
scikit-learn==1.5.2
langchain==0.3.4
langchain-community==0.3.3
langchain-text-splitters==0.3.0
faiss-cpu==1.8.0.post1
sentence-transformers==3.1.1
python-multipart==0.0.9
httpx==0.27.2
structlog==24.4.0

Overwriting /content/drive/MyDrive/isic-flagship-project/requirements.txt


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/.dockerignore
__pycache__/
*.pyc
*.pyo
*.pyd
.Python
env/
venv/
.venv/
.git/
.gitignore
.ipynb_checkpoints/
*.ipynb
logs/

Overwriting /content/drive/MyDrive/isic-flagship-project/.dockerignore


In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
!gcloud projects list

PROJECT_ID              NAME                   PROJECT_NUMBER  ENVIRONMENT
isic-flagship-project   isic-flagship-project  918647643601
starry-arbor-463912-f8  My First Project       1045575864374


In [4]:
PROJECT_ID = "isic-flagship-project"
REGION = "europe-west2"
SERVICE_NAME = "isic-api"

!gcloud config set project {PROJECT_ID}

Updated property [core/project].


In [6]:
PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]

SERVICE_ACCOUNT = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"

print("Project number:", PROJECT_NUMBER)
print("Cloud Run service account:", SERVICE_ACCOUNT)

Project number: 918647643601
Cloud Run service account: 918647643601-compute@developer.gserviceaccount.com


In [7]:
!gcloud secrets add-iam-policy-binding power-automate-url \
  --member="serviceAccount:{SERVICE_ACCOUNT}" \
  --role="roles/secretmanager.secretAccessor" \
  --project={PROJECT_ID}

Updated IAM policy for secret [power-automate-url].
bindings:
- members:
  - serviceAccount:918647643601-compute@developer.gserviceaccount.com
  role: roles/secretmanager.secretAccessor
etag: BwZSFYAoCUM=
version: 1


In [8]:
%cd /content/drive/MyDrive/isic-flagship-project

!gcloud run deploy {SERVICE_NAME} \
  --source . \
  --region {REGION} \
  --allow-unauthenticated \
  --memory 4Gi \
  --cpu 1 \
  --min-instances 1 \
  --max-instances 3 \
  --timeout 300 \
  --port 8080 \
  --cpu-boost \
  --set-secrets POWER_AUTOMATE_URL=power-automate-url:latest \
  --project {PROJECT_ID}

/content/drive/MyDrive/isic-flagship-project
Building using Dockerfile and deploying container to Cloud Run service [isic-api] in project [isic-flagship-project] region [europe-west2]
Service [isic-api] revision [isic-api-00002-l4r] has been deployed and is serving 100 percent of traffic.
Service URL: https://isic-api-918647643601.europe-west2.run.app
